# GFCC-BiLSTM

Gammatone Frequency Cepstral Coefficients (GFCC) fed into a Bidirectional LSTM.
Gammatone filters model the human auditory periphery (basilar membrane), using
ERB-spaced centre frequencies — making GFCC more perceptually motivated than MFCC or LFCC.

## 1. Imports & Config

In [1]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


In [2]:
ROOT_PATH = "/kaggle/input/datasets/arthjs/dataset-ua-asr/dataset_UA_ASR"  # change this

SR = 16000
N_GFCC = 20          # number of GFCC coefficients
FRAME_LEN = 0.025     # 25 ms
HOP_LEN   = 0.010     # 10 ms
N_FILTERS = 64        # gammatone filter channels
LOW_FREQ  = 100.0     # lowest centre frequency (Hz)

FIXED_LEN  = 250      # time frames

# BiLSTM hyper-params
HIDDEN_SIZE = 256
NUM_LAYERS  = 2
DROPOUT     = 0.3

BATCH_SIZE = 32
EPOCHS     = 20
LR         = 1e-3


In [3]:
label_map = {
    "Normal":   0,
    "High":     1,
    "Mid":      2,
    "Low":      3,
    "Very_Low": 4
}
NUM_CLASSES = len(label_map)

In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


## 2. Feature Extraction — GFCC

Pipeline: IIR gammatone filterbank → log frame energy → DCT

In [5]:
from scipy.fftpack import dct
from scipy.signal  import gammatone, lfilter

def erb_space(low_freq, high_freq, n):
    """ERB-spaced centre frequencies (Patterson et al.)."""
    ear_q  = 9.26449
    min_bw = 24.7
    return -(ear_q * min_bw) + np.exp(
        np.arange(1, n+1) *
        (-np.log(high_freq + ear_q*min_bw) + np.log(low_freq + ear_q*min_bw)) / n
    ) * (high_freq + ear_q * min_bw)


def gammatone_filterbank(signal, sr, n_filters, low_freq):
    """Apply IIR gammatone filterbank. Returns (n_filters, n_samples)."""
    cfs    = erb_space(low_freq, sr/2.0, n_filters)
    output = np.zeros((n_filters, len(signal)))
    for i, cf in enumerate(cfs):
        b, a       = gammatone(cf, 'iir', fs=sr)
        output[i]  = np.abs(lfilter(b, a, signal))
    return output


def extract_gfcc(file_path):
    """
    Gammatone Frequency Cepstral Coefficients (GFCC).
    Models the basilar membrane response (auditory periphery).
    Steps: gammatone filterbank → log frame energy → DCT → GFCC
    """
    y, sr      = librosa.load(file_path, sr=SR)
    frame_len  = int(FRAME_LEN * sr)
    hop_len    = int(HOP_LEN   * sr)

    # 1. Gammatone filterbank  (N_FILTERS channels)
    gt = gammatone_filterbank(y, sr, N_FILTERS, LOW_FREQ)  # (N_FILTERS, T)

    # 2. Frame + log energy
    n_frames = 1 + (gt.shape[1] - frame_len) // hop_len
    frames   = np.zeros((N_FILTERS, n_frames))
    for t in range(n_frames):
        s = t * hop_len
        frames[:, t] = np.log(np.sum(gt[:, s:s+frame_len]**2, axis=1) + 1e-10)

    # 3. DCT → N_GFCC coefficients
    gfcc = dct(frames, type=2, axis=0, norm='ortho')[:N_GFCC, :]

    # 4. Fix length
    if gfcc.shape[1] < FIXED_LEN:
        gfcc = np.pad(gfcc, ((0,0),(0,FIXED_LEN-gfcc.shape[1])), mode='constant')
    else:
        gfcc = gfcc[:, :FIXED_LEN]
    return gfcc   # (N_GFCC, FIXED_LEN)


## 3. Precompute & Save Features

In [6]:
SAVE_PATH = "/kaggle/working/gfcc_features"
os.makedirs(SAVE_PATH, exist_ok=True)

def save_gfcc_dataset(root_dir, split):
    split_path = os.path.join(root_dir, split)
    for severity in os.listdir(split_path):
        sev_path = os.path.join(split_path, severity)
        if not os.path.isdir(sev_path): continue
        save_sev = os.path.join(SAVE_PATH, split, severity)
        os.makedirs(save_sev, exist_ok=True)
        for file in tqdm(os.listdir(sev_path), desc=f'{split}-{severity}'):
            if not file.endswith('.flac'): continue
            np.save(os.path.join(save_sev, file.replace('.flac','.npy')),
                    extract_gfcc(os.path.join(sev_path, file)))

save_gfcc_dataset(ROOT_PATH, 'train')
save_gfcc_dataset(ROOT_PATH, 'test')


test-High: 100%|██████████| 5425/5425 [04:50<00:00, 18.68it/s]


## 4. Dataset & DataLoaders

In [7]:
import torch
from torch.utils.data import Dataset

class GfccDataset(Dataset):
    def __init__(self, root_dir):
        self.files  = []
        self.labels = []
        for severity in os.listdir(root_dir):
            sev_path = os.path.join(root_dir, severity)
            if not os.path.isdir(sev_path): continue
            for file in os.listdir(sev_path):
                if file.endswith('.npy'):
                    self.files.append(os.path.join(sev_path, file))
                    self.labels.append(label_map[severity])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        gfcc = np.load(self.files[idx])          # (N_FEAT, FIXED_LEN)
        # BiLSTM expects (seq_len, input_size) → transpose to (FIXED_LEN, N_FEAT)
        gfcc = torch.tensor(gfcc.T, dtype=torch.float32)  # (FIXED_LEN, N_FEAT)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return gfcc, label


In [8]:
train_dataset = GfccDataset("/kaggle/working/gfcc_features/train")
test_dataset  = GfccDataset("/kaggle/working/gfcc_features/test")

val_size   = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_ds, val_ds = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 5. BiLSTM Model

In [9]:
import torch
import torch.nn as nn

drop_amount = 0.255

class BiLSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=drop_amount if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(drop_amount)

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):

        # Initial hidden and cell states
        h0 = torch.zeros(
            self.num_layers * 2,
            x.size(0),
            self.hidden_size,
            device=x.device
        )

        c0 = torch.zeros(
            self.num_layers * 2,
            x.size(0),
            self.hidden_size,
            device=x.device
        )

        # LSTM output
        out, _ = self.lstm(x, (h0, c0))

        out = self.dropout(out)

        # Forward last timestep + backward first timestep
        last_hidden_state = torch.cat(
            (
                out[:, -1, :self.hidden_size],
                out[:, 0, self.hidden_size:]
            ),
            dim=1
        )

        output = self.fc(last_hidden_state)

        return output

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMClassifier(
    input_size=N_GFCC,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES
).to(device)


## 6. Training

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

import shutil

best_val_acc = 0
save_path = "/tmp/best_model.pth"

for epoch in range(EPOCHS):

    # ================= TRAIN =================
    model.train()
    train_loss = 0

    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [train]'):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    # ================= VALIDATION =================
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == y).sum().item()

            total += y.size(0)

    val_acc = correct / total

    print(f'Epoch {epoch+1}/{EPOCHS}: Loss={train_loss:.4f}  Val Acc={val_acc:.4f}')

    # ================= SAVE BEST =================
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            save_path,
            _use_new_zipfile_serialization=False
        )

        print(f'✅ Best model saved (Val Acc={val_acc:.4f})')

shutil.copy(save_path, './best_model.pth')

print('✅ Model copied to working directory!')

Model params: 2,148,869


Epoch 1/20 [train]: 100%|██████████| 2151/2151 [01:53<00:00, 18.95it/s]


Epoch 1/20: Loss=425.3692  Val Acc=0.9800
✅ Best model saved (Val Acc=0.9800)


Epoch 2/20 [train]: 100%|██████████| 2151/2151 [02:00<00:00, 17.85it/s]


Epoch 2/20: Loss=138.3400  Val Acc=0.9840
✅ Best model saved (Val Acc=0.9840)


...

Epoch 18/20: Loss=30.4391  Val Acc=0.9935


Epoch 19/20 [train]: 100%|██████████| 2151/2151 [02:00<00:00, 17.90it/s]


Epoch 19/20: Loss=26.5775  Val Acc=0.9957
✅ Best model saved (Val Acc=0.9957)


Epoch 20/20 [train]: 100%|██████████| 2151/2151 [02:00<00:00, 17.89it/s]


Epoch 20/20: Loss=27.0645  Val Acc=0.9952
✅ Model copied to working directory!


## 7. Evaluation

In [12]:
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

correct = total = 0
all_preds  = []
all_labels = []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc='Test'):
        x, y  = x.to(device), y.to(device)
        preds = torch.argmax(model(x), dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

print(f'✅ Test Accuracy: {correct/total:.4f}')


Test: 100%|██████████| 1185/1185 [00:24<00:00, 47.83it/s]

✅ Test Accuracy: 0.9790


In [13]:
from sklearn.metrics import classification_report
print('\nClassification Report:\n')
print(classification_report(all_labels, all_preds, target_names=list(label_map.keys())))



Classification Report:

              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00     23205
        High       0.96      0.98      0.97      5425
         Mid       0.96      0.90      0.93      3100
         Low       0.99      0.92      0.95      3100
    Very_Low       0.91      0.97      0.94      3086

    accuracy                           0.98     37916
   macro avg       0.96      0.95      0.96     37916
weighted avg       0.98      0.98      0.98     37916



In [14]:
# import pandas as pd

# inv_map = {v: k for k, v in label_map.items()}

# filenames = [os.path.basename(test_dataset.files[i]) for i in range(len(test_dataset))]

# scores = []
# all_preds = []
# all_labels = []

# model.eval()
# with torch.no_grad():
#     for x, y in tqdm(DataLoader(test_dataset, batch_size=32, shuffle=False)):
#         x, y = x.to(device), y.to(device)
#         outputs = model(x)
#         probs = torch.softmax(outputs, dim=1)
#         best_scores = probs.max(dim=1).values
#         preds = torch.argmax(outputs, dim=1)

#         scores.extend(best_scores.cpu().numpy())
#         all_preds.extend(preds.cpu().numpy())
#         all_labels.extend(y.cpu().numpy())

# df = pd.DataFrame({
#     "filename":      filenames,
#     "score":         scores,
#     "predict class": [inv_map[p] for p in all_preds],
#     "actual class":  [inv_map[l] for l in all_labels],
# })

# df.to_excel("/kaggle/working/results.xlsx", index=False)
# print("✅ Saved results.xlsx")

import pandas as pd

inv_map = {v: k for k, v in label_map.items()}

# Class order: Normal → High → Mid → Low → Very_Low
class_order = ["Normal", "High", "Mid", "Low", "Very_Low"]

filenames  = [os.path.basename(test_dataset.files[i]) for i in range(len(test_dataset))]

all_preds      = []
all_labels     = []
all_raw_scores = []   # raw logits per class
all_softmax    = []   # softmax probabilities per class

model.eval()
with torch.no_grad():
    for x, y in tqdm(DataLoader(test_dataset, batch_size=32, shuffle=False)):
        x, y = x.to(device), y.to(device)
        outputs = model(x)                          # raw logits: (batch, NUM_CLASSES)
        probs   = torch.softmax(outputs, dim=1)     # softmax probabilities
        preds   = torch.argmax(outputs, dim=1)

        all_raw_scores.extend(outputs.cpu().numpy())
        all_softmax.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Build DataFrame
df = pd.DataFrame({"filename": filenames})

# Raw logit score for each class
for cls_name in class_order:
    df[f"score_{cls_name}"] = [row[label_map[cls_name]] for row in all_raw_scores]

# Softmax probability for each class
for cls_name in class_order:
    df[f"prob_{cls_name}"] = [row[label_map[cls_name]] for row in all_softmax]

# Final prediction & actual label
df["predict class"] = [inv_map[p] for p in all_preds]
df["actual class"]  = [inv_map[l] for l in all_labels]

# Save both CSV and Excel
df.to_csv("/kaggle/working/results.csv",   index=False)
df.to_excel("/kaggle/working/results.xlsx", index=False)
print("✅ Saved results.csv and results.xlsx")
print(df.head())


100%|██████████| 1185/1185 [00:24<00:00, 48.46it/s]


✅ Saved results.csv and results.xlsx
              filename  score_Normal  score_High  score_Mid  score_Low  \
0   CM13_B3_CW9_M6.npy     15.402400   -4.645810  -9.518134  -4.969928   
1  CM09_B3_UW27_M2.npy     14.361671   -3.587114  -8.114631  -5.752865   
2   CM12_B3_CW6_M4.npy     15.417197   -4.787978  -9.501626  -5.908000   
3  CM09_B3_UW78_M8.npy     15.670704   -5.133141  -9.511785  -5.926474   
4  CM08_B3_UW15_M4.npy     13.654040   -2.969773  -7.634202  -4.737118   

   score_Very_Low  prob_Normal     prob_High      prob_Mid      prob_Low  \
0       -9.610847          1.0  1.964142e-09  1.503660e-11  1.420399e-09   
1       -8.833294          1.0  1.603029e-08  1.732469e-10  1.838095e-09   
2       -8.937449          1.0  1.678819e-09  1.506234e-11  5.477527e-10   
3       -9.132032          1.0  9.225824e-10  1.157137e-11  4.173160e-10   
4       -8.984421          1.0  6.030724e-08  5.683735e-10  1.029962e-08   

   prob_Very_Low predict class actual class  
0   1.370516e-1

## 8. Single-File Prediction

In [15]:
# def predict(file_path, model_path='best_model.pth'):
#     m = BiLSTMModel(N_GFCC, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT)
#     m.load_state_dict(torch.load(model_path, map_location='cpu'))
#     m.eval()
#     feat = torch.tensor(extract_gfcc(file_path).T, dtype=torch.float32).unsqueeze(0)
#     with torch.no_grad():
#         pred = torch.argmax(m(feat), dim=1).item()
#     return {v:k for k,v in label_map.items()}[pred]
